# Atelier Préparatione Données Images

Contexte

Une entreprise souhaite développer un système d’intelligence artificielle capable de reconnaître
automatiquement le type de déchet présent sur une photographie afin d'améliorer le tri des
déchets.

Le modèle devra classer chaque image dans l'une des catégories suivantes :

 cardboard : cartons ondulés, cartons plats, …
 plastic : bouteilles, emballages plastiques...
 paper : feuilles, journaux...
 glass : bouteilles et objets en verre...
 metal : canettes, boîtes métalliques...
 trash : emballages bonbons, tasses jetables, ...

Le problème est que les images collectées proviennent de plusieurs sources. Elles ne sont donc pas
homogènes : dimensions différentes ; formats différents ; images RGB et grayscale ; certaines images
sont trop petites ; certaines images sont corrompues ; quelques images sont vides ; images
dupliquées ; quelques images placées dans le mauvais dossier ; classes déséquilibrées.
L'objectif de l'atelier est donc de construire un jeu de données images propre et homogène, prêt à
être utilisé pour entraîner un modèle de Machine Learning ou de Deep Learning.

Objectifs pédagogiques

À la fin de l'atelier, l'apprenant devra être capable de :
1) explorer un dataset d'images ;
2) détecter les images problématiques ;
3) détecter les différences de résolution ;
4) détecter les différences de nombre de canaux ;
5) identifier les images trop petites ;
6) détecter les doublons ;
7) identifier les classes déséquilibrées ;
8) redimensionner les images ;
9) normaliser les valeurs des pixels ;
10) uniformiser les canaux ;
11) appliquer de la data augmentation

In [1]:
import PIL, numpy, pandas, matplotlib, imagehash, sklearn, tensorflow
print("Tout est installé correctement")

Tout est installé correctement


# Partie 1 – Exploration du dataset

##   lister les classes du dataset

In [2]:
import os  # module natif Python pour interagir avec les fichiers et dossiers du système

dossier_raw = "../data/raw"  # chemin vers le dossier qui contient les 6 sous-dossiers de classes
classes = os.listdir(dossier_raw)  # liste les noms des sous-dossiers = liste des classes

print(classes)  # affiche la liste des classes trouvées

['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']


## lister le nom de chaque image 

In [7]:
import os  # module natif Python pour parcourir les fichiers et dossiers du systeme

dossier_raw = "../data/raw"  # chemin vers le dossier qui contient les 6 sous-dossiers de classes
classes = os.listdir(dossier_raw)  # liste les noms des sous-dossiers (= les classes)

noms_images = []  # liste vide qui va accueillir le nom de chaque image trouvee

for classe in classes:  # on parcourt chaque classe, une par une
    chemin_classe = os.path.join(dossier_raw, classe)  # chemin complet vers le dossier de cette classe
    for nom_fichier in os.listdir(chemin_classe):  # on parcourt chaque fichier image present dans ce dossier
        noms_images.append(nom_fichier)  # on ajoute le nom de ce fichier a la liste

print("Nombre total d'images :", len(noms_images))  # verification du nombre total de noms recuperes
print(noms_images[:5])  # affiche les 5 premiers noms, pour verifier a quoi ils ressemblent

Nombre total d'images : 1032
['cardboard1.jpg', 'cardboard10.jpg', 'cardboard100.jpg', 'cardboard101.jpg', 'cardboard102.jpg']


##  format de chaque image

In [11]:
infos_format = []  # liste qui va garder nom ET format ensemble, pour chaque image

for classe in classes:  # on parcourt chaque classe
    chemin_classe = os.path.join(dossier_raw, classe)  # chemin vers le dossier de cette classe
    for nom_fichier in os.listdir(chemin_classe):  # on parcourt chaque fichier de cette classe
        chemin_complet = os.path.join(chemin_classe, nom_fichier)  # chemin complet vers l'image
        try:  # on tente d'ouvrir l'image
            img = Image.open(chemin_complet)  # ouverture avec Pillow
            infos_format.append({"nom": nom_fichier, "format": img.format})  # on garde nom + format lies ensemble
        except Exception as e:  # si l'ouverture echoue
            pass  # on ignore ce fichier pour l'instant (deja compte comme corrompu avant)

print(infos_format[:5])  # affiche les 5 premieres paires nom/format, pour VOIR la correspondance

[{'nom': 'cardboard1.jpg', 'format': 'JPEG'}, {'nom': 'cardboard10.jpg', 'format': 'JPEG'}, {'nom': 'cardboard100.jpg', 'format': 'JPEG'}, {'nom': 'cardboard101.jpg', 'format': 'JPEG'}, {'nom': 'cardboard102.jpg', 'format': 'JPEG'}]


## recuperer le mode de chaque image

In [14]:
infos_mode = []  # liste qui va garder nom ET mode ensemble, pour chaque image

for classe in classes:  # on parcourt chaque classe
    chemin_classe = os.path.join(dossier_raw, classe)  # chemin vers le dossier de cette classe
    for nom_fichier in os.listdir(chemin_classe):  # on parcourt chaque fichier de cette classe
        chemin_complet = os.path.join(chemin_classe, nom_fichier)  # chemin complet vers l'image
        try:  # on tente d'ouvrir l'image
            img = Image.open(chemin_complet)  # ouverture avec Pillow
            infos_mode.append({"nom": nom_fichier, "mode": img.mode})  # on garde nom + mode lies ensemble
        except Exception as e:  # si l'ouverture echoue (fichier corrompu)
            pass  # on ignore ce fichier, deja compte comme corrompu a la micro-tache precedente

print("Nombre de modes recuperes :", len(infos_mode))  # verification du nombre total
print(infos_mode[:5])  # affiche les 5 premieres paires nom/mode, pour voir la correspondance

Nombre de modes recuperes : 1026
[{'nom': 'cardboard1.jpg', 'mode': 'RGB'}, {'nom': 'cardboard10.jpg', 'mode': 'RGB'}, {'nom': 'cardboard100.jpg', 'mode': 'RGB'}, {'nom': 'cardboard101.jpg', 'mode': 'RGB'}, {'nom': 'cardboard102.jpg', 'mode': 'RGB'}]


## recuperer largeur et hauteur de chaque image

In [15]:
infos_dimensions = []  # liste qui va garder nom, largeur et hauteur ensemble, pour chaque image

for classe in classes:  # on parcourt chaque classe
    chemin_classe = os.path.join(dossier_raw, classe)  # chemin vers le dossier de cette classe
    for nom_fichier in os.listdir(chemin_classe):  # on parcourt chaque fichier de cette classe
        chemin_complet = os.path.join(chemin_classe, nom_fichier)  # chemin complet vers l'image
        try:  # on tente d'ouvrir l'image
            img = Image.open(chemin_complet)  # ouverture avec Pillow
            largeur, hauteur = img.size  # .size renvoie un tuple (largeur, hauteur) en pixels
            infos_dimensions.append({"nom": nom_fichier, "largeur": largeur, "hauteur": hauteur})  # on garde les 3 infos liees ensemble
        except Exception as e:  # si l'ouverture echoue (fichier corrompu)
            pass  # on ignore ce fichier, deja compte comme corrompu

print("Nombre d'images mesurees :", len(infos_dimensions))  # verification du nombre total
print(infos_dimensions[:5])  # affiche les 5 premieres lignes, pour voir la correspondance nom/largeur/hauteur

Nombre d'images mesurees : 1026
[{'nom': 'cardboard1.jpg', 'largeur': 512, 'hauteur': 384}, {'nom': 'cardboard10.jpg', 'largeur': 512, 'hauteur': 384}, {'nom': 'cardboard100.jpg', 'largeur': 512, 'hauteur': 384}, {'nom': 'cardboard101.jpg', 'largeur': 512, 'hauteur': 384}, {'nom': 'cardboard102.jpg', 'largeur': 512, 'hauteur': 384}]


## calculer l'ecart-type des pixels de chaque image

In [16]:
import numpy as np  # bibliotheque pour manipuler des tableaux de nombres, necessaire pour calculer un ecart-type

infos_ecart_type = []  # liste qui va garder nom et ecart-type ensemble, pour chaque image

for classe in classes:  # on parcourt chaque classe
    chemin_classe = os.path.join(dossier_raw, classe)  # chemin vers le dossier de cette classe
    for nom_fichier in os.listdir(chemin_classe):  # on parcourt chaque fichier de cette classe
        chemin_complet = os.path.join(chemin_classe, nom_fichier)  # chemin complet vers l'image
        try:  # on tente d'ouvrir et de traiter l'image
            img = Image.open(chemin_complet)  # ouverture avec Pillow
            tableau_pixels = np.array(img)  # convertit l'image en tableau numpy de valeurs de pixels (0 a 255)
            ecart_type = tableau_pixels.std()  # calcule l'ecart-type de toutes les valeurs de pixels
            infos_ecart_type.append({"nom": nom_fichier, "ecart_type": ecart_type})  # on garde nom + ecart-type lies ensemble
        except Exception as e:  # si l'ouverture echoue (fichier corrompu)
            pass  # on ignore ce fichier, deja compte comme corrompu

print("Nombre d'images traitees :", len(infos_ecart_type))  # verification du nombre total
print(infos_ecart_type[:5])  # affiche les 5 premieres lignes, pour voir la correspondance nom/ecart-type

Nombre d'images traitees : 1026
[{'nom': 'cardboard1.jpg', 'ecart_type': np.float64(40.58852860499886)}, {'nom': 'cardboard10.jpg', 'ecart_type': np.float64(42.57128838691107)}, {'nom': 'cardboard100.jpg', 'ecart_type': np.float64(46.108304565445664)}, {'nom': 'cardboard101.jpg', 'ecart_type': np.float64(72.26399578522313)}, {'nom': 'cardboard102.jpg', 'ecart_type': np.float64(48.3889369058145)}]


## recuperer le nombre de canaux de chaque image

In [17]:
infos_canaux = []  # liste qui va garder nom et nombre de canaux ensemble, pour chaque image

for classe in classes:  # on parcourt chaque classe
    chemin_classe = os.path.join(dossier_raw, classe)  # chemin vers le dossier de cette classe
    for nom_fichier in os.listdir(chemin_classe):  # on parcourt chaque fichier de cette classe
        chemin_complet = os.path.join(chemin_classe, nom_fichier)  # chemin complet vers l'image
        try:  # on tente d'ouvrir l'image
            img = Image.open(chemin_complet)  # ouverture avec Pillow
            nombre_canaux = len(img.getbands())  # getbands() renvoie un tuple des canaux presents (ex: ('R','G','B')), on compte sa longueur
            infos_canaux.append({"nom": nom_fichier, "nombre_canaux": nombre_canaux})  # on garde nom + nombre de canaux lies ensemble
        except Exception as e:  # si l'ouverture echoue (fichier corrompu)
            pass  # on ignore ce fichier, deja compte comme corrompu

print("Nombre d'images traitees :", len(infos_canaux))  # verification du nombre total
print(infos_canaux[:5])  # affiche les 5 premieres lignes, pour voir la correspondance nom/nombre_canaux

Nombre d'images traitees : 1026
[{'nom': 'cardboard1.jpg', 'nombre_canaux': 3}, {'nom': 'cardboard10.jpg', 'nombre_canaux': 3}, {'nom': 'cardboard100.jpg', 'nombre_canaux': 3}, {'nom': 'cardboard101.jpg', 'nombre_canaux': 3}, {'nom': 'cardboard102.jpg', 'nombre_canaux': 3}]


## recuperer la taille de chaque fichier image

In [18]:
infos_taille = []  # liste qui va garder nom et taille du fichier ensemble, pour chaque image

for classe in classes:  # on parcourt chaque classe
    chemin_classe = os.path.join(dossier_raw, classe)  # chemin vers le dossier de cette classe
    for nom_fichier in os.listdir(chemin_classe):  # on parcourt chaque fichier de cette classe
        chemin_complet = os.path.join(chemin_classe, nom_fichier)  # chemin complet vers l'image
        try:  # on tente d'ouvrir l'image (pour rester coherent avec les autres micro-taches, meme si getsize ne l'exige pas)
            img = Image.open(chemin_complet)  # ouverture avec Pillow
            taille_octets = os.path.getsize(chemin_complet)  # taille du fichier sur le disque, en octets
            infos_taille.append({"nom": nom_fichier, "taille_octets": taille_octets})  # on garde nom + taille liees ensemble
        except Exception as e:  # si l'ouverture echoue (fichier corrompu)
            pass  # on ignore ce fichier, deja compte comme corrompu

print("Nombre d'images traitees :", len(infos_taille))  # verification du nombre total
print(infos_taille[:5])  # affiche les 5 premieres lignes, pour voir la correspondance nom/taille

Nombre d'images traitees : 1026
[{'nom': 'cardboard1.jpg', 'taille_octets': 17333}, {'nom': 'cardboard10.jpg', 'taille_octets': 21683}, {'nom': 'cardboard100.jpg', 'taille_octets': 14884}, {'nom': 'cardboard101.jpg', 'taille_octets': 14289}, {'nom': 'cardboard102.jpg', 'taille_octets': 18015}]


## recuperer le chemin complet de chaque imag

In [19]:
infos_chemin = []  # liste qui va garder nom et chemin complet ensemble, pour chaque image

for classe in classes:  # on parcourt chaque classe
    chemin_classe = os.path.join(dossier_raw, classe)  # chemin vers le dossier de cette classe
    for nom_fichier in os.listdir(chemin_classe):  # on parcourt chaque fichier de cette classe
        chemin_complet = os.path.join(chemin_classe, nom_fichier)  # chemin complet vers l'image (dossier + nom du fichier)
        infos_chemin.append({"nom": nom_fichier, "chemin": chemin_complet})  # on garde nom + chemin lies ensemble

print("Nombre de chemins recuperes :", len(infos_chemin))  # verification du nombre total
print(infos_chemin[:5])  # affiche les 5 premieres lignes, pour voir la correspondance nom/chemin

Nombre de chemins recuperes : 1032
[{'nom': 'cardboard1.jpg', 'chemin': '../data/raw\\cardboard\\cardboard1.jpg'}, {'nom': 'cardboard10.jpg', 'chemin': '../data/raw\\cardboard\\cardboard10.jpg'}, {'nom': 'cardboard100.jpg', 'chemin': '../data/raw\\cardboard\\cardboard100.jpg'}, {'nom': 'cardboard101.jpg', 'chemin': '../data/raw\\cardboard\\cardboard101.jpg'}, {'nom': 'cardboard102.jpg', 'chemin': '../data/raw\\cardboard\\cardboard102.jpg'}]


## construction du DataFrame complet (nom, classe, format, mode, dimensions, ecart-type, canaux, taille)

In [20]:
import pandas as pd  # bibliotheque pour construire et manipuler des tableaux de donnees (DataFrame)

donnees_images = []  # liste vide qui va accueillir un dictionnaire complet par image
fichiers_corrompus = []  # liste vide qui va accueillir les noms des fichiers impossibles a ouvrir

for classe in classes:  # on parcourt chaque classe, une par une
    chemin_classe = os.path.join(dossier_raw, classe)  # chemin complet vers le dossier de cette classe
    for nom_fichier in os.listdir(chemin_classe):  # on parcourt chaque fichier de cette classe
        chemin_complet = os.path.join(chemin_classe, nom_fichier)  # chemin complet vers cette image precise

        try:  # on tente de tout recuperer pour cette image, en une seule ouverture
            img = Image.open(chemin_complet)  # ouverture de l'image avec Pillow (une seule fois, pas 7 fois)
            tableau_pixels = np.array(img)  # conversion en tableau numpy pour calculer l'ecart-type

            donnees_images.append({
                "nom": nom_fichier,                      # nom du fichier
                "classe": classe,                        # classe = dossier parent
                "chemin": chemin_complet,                # chemin complet vers le fichier
                "format": img.format,                    # format du fichier (ex: JPEG)
                "mode": img.mode,                         # mode de couleur (ex: RGB, L, RGBA)
                "largeur": img.size[0],                  # largeur en pixels
                "hauteur": img.size[1],                  # hauteur en pixels
                "ecart_type": tableau_pixels.std(),      # ecart-type des valeurs de pixels
                "nombre_canaux": len(img.getbands()),    # nombre de canaux de couleur
                "taille_octets": os.path.getsize(chemin_complet)  # taille du fichier sur le disque
            })

        except Exception as e:  # si l'ouverture ou le traitement echoue (fichier corrompu)
            fichiers_corrompus.append(nom_fichier)  # on note juste le nom du fichier problematique

df_images = pd.DataFrame(donnees_images)  # transforme la liste de dictionnaires en un vrai tableau pandas

print("Images traitees avec succes :", len(df_images))  # nombre de lignes dans le tableau final
print("Fichiers corrompus ecartes :", len(fichiers_corrompus))  # nombre de fichiers qui ont echoue
df_images.head()  # affiche les 5 premieres lignes du tableau, avec toutes les colonnes

Images traitees avec succes : 1026
Fichiers corrompus ecartes : 6


,nom,classe,chemin,format,mode,largeur,hauteur,ecart_type,nombre_canaux,taille_octets
0,cardboard1.jpg,cardboard,../data/raw\cardboard\cardboard1.jpg,JPEG,RGB,512,384,40.588529,3,17333
1,cardboard10.jpg,cardboard,../data/raw\cardboard\cardboard10.jpg,JPEG,RGB,512,384,42.571288,3,21683
2,cardboard100.jpg,cardboard,../data/raw\cardboard\cardboard100.jpg,JPEG,RGB,512,384,46.108305,3,14884
3,cardboard101.jpg,cardboard,../data/raw\cardboard\cardboard101.jpg,JPEG,RGB,512,384,72.263996,3,14289
4,cardboard102.jpg,cardboard,../data/raw\cardboard\cardboard102.jpg,JPEG,RGB,512,384,48.388937,3,18015


# Partie 2 – Détecter les images corrompues

## fontion est_corrompu 

In [21]:
def est_corrompue(chemin_image):  # definit une fonction reutilisable, qui prend un chemin d'image en parametre
    try:  # on tente d'ouvrir ET de verifier l'integrite de l'image
        img = Image.open(chemin_image)  # ouverture de l'image avec Pillow
        img.verify()  # verify() controle que le fichier est structurellement valide, sans le decoder entierement
        return False  # si aucune erreur n'est levee, l'image n'est pas corrompue
    except Exception as e:  # si une erreur survient a l'ouverture ou a la verification
        return True  # l'image est consideree comme corrompue

## Partie2: application de est_corrompue sur tout le dataset

In [22]:
resultats_corruption = []  # liste vide qui va accueillir le nom de chaque image et si elle est corrompue ou non

for classe in classes:  # on parcourt chaque classe, une par une
    chemin_classe = os.path.join(dossier_raw, classe)  # chemin complet vers le dossier de cette classe
    for nom_fichier in os.listdir(chemin_classe):  # on parcourt chaque fichier de cette classe
        chemin_complet = os.path.join(chemin_classe, nom_fichier)  # chemin complet vers cette image precise
        corrompue = est_corrompue(chemin_complet)  # on appelle la fonction definie juste avant, qui renvoie True ou False
        resultats_corruption.append({"nom": nom_fichier, "classe": classe, "corrompue": corrompue})  # on garde le resultat pour cette image

df_corruption = pd.DataFrame(resultats_corruption)  # transforme la liste en tableau pandas

nb_corrompues = df_corruption["corrompue"].sum()  # compte le nombre de True (True vaut 1, False vaut 0)
print("Nombre total d'images corrompues :", nb_corrompues)  # affiche le total
print(df_corruption[df_corruption["corrompue"] == True])  # affiche uniquement les lignes ou corrompue est True

Nombre total d'images corrompues : 6
                  nom     classe  corrompue
147   cardboard83.jpg  cardboard       True
326       glass74.jpg      glass       True
446       metal48.jpg      metal       True
633      paper213.jpg      paper       True
791     plastic13.jpg    plastic       True
1004       trash3.jpg      trash       True


# Partie 3 – Détecter les images vides

## nction est_vide basee sur l'ecart-type des pixels

In [24]:
def est_vide(chemin_image, seuil=10):  # definit une fonction reutilisable ; seuil = valeur en dessous de laquelle on considere l'image comme vide
    try:  # on tente d'ouvrir et d'analyser l'image
        img = Image.open(chemin_image)  # ouverture de l'image avec Pillow
        tableau_pixels = np.array(img)  # conversion en tableau numpy de valeurs de pixels
        ecart_type = tableau_pixels.std()  # calcule la variation des valeurs de pixels (deja vu en Partie 1)
        return ecart_type < seuil  # renvoie True si la variation est tres faible (image quasi uniforme)
    except Exception as e:  # si l'image est corrompue et ne peut pas etre analysee
        return False  # on ne la compte pas comme "vide", elle est deja comptee comme corrompue ailleurs

##  application de est_vide sur tout le dataset

In [25]:
resultats_vide = []  # liste vide qui va accueillir le nom de chaque image et si elle est vide ou non

for classe in classes:  # on parcourt chaque classe, une par une
    chemin_classe = os.path.join(dossier_raw, classe)  # chemin complet vers le dossier de cette classe
    for nom_fichier in os.listdir(chemin_classe):  # on parcourt chaque fichier de cette classe
        chemin_complet = os.path.join(chemin_classe, nom_fichier)  # chemin complet vers cette image precise
        vide = est_vide(chemin_complet)  # on appelle la fonction definie juste avant, qui renvoie True ou False
        resultats_vide.append({"nom": nom_fichier, "classe": classe, "vide": vide})  # on garde le resultat pour cette image

df_vide = pd.DataFrame(resultats_vide)  # transforme la liste en tableau pandas

nb_vides = df_vide["vide"].sum()  # compte le nombre de True (True vaut 1, False vaut 0)
print("Nombre total d'images quasi vides :", nb_vides)  # affiche le total
print(df_vide[df_vide["vide"] == True])  # affiche uniquement les lignes ou vide est True

Nombre total d'images quasi vides : 4
                           nom     classe  vide
167  image-blanche-512x384.jpg  cardboard  True
357  image-blanche-512x384.jpg      metal  True
815             plastic150.jpg    plastic  True
902               plastic3.jpg    plastic  True


# Partie 4 – Détecter les différences de résolution

##  1 Resolution

### a: image et resolution minimale

In [34]:
df_images["aire"] = df_images["largeur"] * df_images["hauteur"]  # calcule le nombre total de pixels de chaque image (largeur fois hauteur)

image_plus_petite = df_images.loc[df_images["aire"].idxmin()]  # idxmin() trouve l'index de la ligne avec l'aire la plus petite ; .loc recupere cette ligne entiere
resolution_min = f"{image_plus_petite['largeur']}*{image_plus_petite['hauteur']}"  # construit le texte de la resolution, ex: "50x60"

print("Image avec la plus petite resolution :")  # titre
print(image_plus_petite[["nom", "classe", "largeur", "hauteur", "aire"]])  # affiche nom, classe, largeur et hauteur de cette image precise
print("Resolution :", resolution_min)  # affiche la resolution sous forme combinee

Image avec la plus petite resolution :
nom        cardboard22.jpg
classe           cardboard
largeur                 32
hauteur                 32
aire                  1024
Name: 77, dtype: object
Resolution : 32*32


### b: image et resolution maximale

In [36]:
image_plus_grande = df_images.loc[df_images["aire"].idxmax()]  # idxmax() trouve l'index de la ligne avec l'aire la plus grande ; .loc recupere cette ligne entiere
resolution_max = f"{image_plus_grande['largeur']}x{image_plus_grande['hauteur']}"  # construit le texte de la resolution, ex: "1920x1080"

print("Image avec la plus grande resolution :")  # titre
print(image_plus_grande[["nom", "classe", "largeur", "hauteur", "aire"]])  # affiche nom, classe, largeur et hauteur de cette image precise
print("Resolution :", resolution_max)  # affiche la resolution sous forme combinee

Image avec la plus grande resolution :
nom        cardboard1.jpg
classe          cardboard
largeur               512
hauteur               384
aire               196608
Name: 0, dtype: object
Resolution : 512x384


### c: resolutions les plus frequentes

In [38]:
comptage_resolutions = df_images.groupby(["largeur", "hauteur"]).size()  # groupe les lignes par paire (largeur, hauteur) identique, et compte combien de lignes dans chaque groupe
comptage_resolutions = comptage_resolutions.sort_values(ascending=False)  # trie du plus frequent au moins frequent

print(comptage_resolutions.head(10))  # affiche les 10 plus frequentes

largeur  hauteur
512      384        1013
32       32            5
40       40            4
48       32            4
dtype: int64


###  1d: inventaire complet du nombre d'images par resolution

In [39]:
comptage_resolutions = df_images.groupby(["largeur", "hauteur"]).size()  # groupe les images par paire (largeur, hauteur) identique, compte combien de lignes dans chaque groupe

print("Nombre d'images pour chaque resolution existante :")  # titre
print(comptage_resolutions)  # affiche TOUT le tableau, sans filtrage, chaque resolution existante avec son nombre d'images

Nombre d'images pour chaque resolution existante :
largeur  hauteur
32       32            5
40       40            4
48       32            4
512      384        1013
dtype: int64
